In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/02_api_football_client

In [0]:
import json
from pathlib import Path
from typing import Any


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide league IDs, seasons, API settings, secrets, paths, and daily
# quota configuration are loaded from 00_project_config.
#
# HTTP communication, retries, request pacing, quota tracking, error handling,
# and atomic JSON writes are provided by 02_api_football_client.

ENDPOINT = "teams"

RAW_FILES_ROOT = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/team_data/league_season"
)

CONSOLIDATED_OUTPUT_FILE = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/team_data/raw_team_data.json"
)

# Leave a small number of requests unused as a safety margin.
#
# The actual remaining quota reported by API-Football takes precedence once
# requests begin.
MAX_REQUESTS_PER_RUN = (
    API_FOOTBALL_DAILY_REQUEST_LIMIT
    - API_FOOTBALL_DAILY_REQUEST_RESERVE
)


# -----------------------------------------------------------------------------
# Raw-file helpers
# -----------------------------------------------------------------------------

def get_raw_file_path(
    league_id: int,
    season: int,
) -> Path:
    """
    Return the raw file path for one league-season teams response.

    Each successful API request is persisted separately so the ingestion can
    resume safely without repeating completed league-season requests.
    """

    return Path(
        RAW_FILES_ROOT,
        f"league_id={league_id}",
        f"season={season}",
        "teams.json",
    )


def read_json_file(
    file_path: Path,
) -> dict[str, Any]:
    """Read and validate one persisted raw JSON response."""

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError(
            f"Expected JSON object in raw file: {file_path}"
        )

    return payload


def validate_teams_payload(
    payload: dict[str, Any],
    *,
    league_id: int,
    season: int,
) -> list[dict[str, Any]]:
    """
    Validate the minimum API-Football structure required by this ingestion.

    Returns the list of team records from the API response.
    """

    teams = payload.get("response")

    if not isinstance(teams, list):
        raise APIFootballError(
            "Expected API-Football 'response' to be a list for "
            f"league={league_id}, season={season}."
        )

    # For the configured top-five leagues and seasons, an empty response is
    # unexpected. Do not persist it because a transient upstream issue could
    # otherwise look like a successfully completed extraction.
    if not teams:
        raise APIFootballError(
            "API-Football returned zero teams for "
            f"league={league_id}, season={season}."
        )

    return teams


def league_season_is_complete(
    league_id: int,
    season: int,
) -> bool:
    """
    Return True when a valid raw teams response already exists locally.
    """

    raw_file = get_raw_file_path(
        league_id,
        season,
    )

    if not raw_file.exists():
        return False

    payload = read_json_file(
        raw_file
    )

    validate_teams_payload(
        payload,
        league_id=league_id,
        season=season,
    )

    return True


def all_extractions_complete() -> bool:
    """
    Return True when every configured league-season response exists locally.
    """

    return all(
        league_season_is_complete(
            league_id,
            season,
        )
        for league_id in API_FOOTBALL_LEAGUES
        for season in API_FOOTBALL_SEASONS
    )


# -----------------------------------------------------------------------------
# Consolidation helper
# -----------------------------------------------------------------------------

def build_consolidated_team_file() -> int:
    """
    Build the existing consolidated team file from persisted raw responses.

    No API requests are made by this function.

    The full source response for each league-season remains available under
    RAW_FILES_ROOT for lineage and debugging.

    The consolidated output preserves the structure currently expected by the
    Bronze ingestion notebook:

        {
            "league_id": ...,
            "league_name": ...,
            "season": ...,
            "data": ...
        }

    Returns:
        Number of team-season records written.
    """

    all_teams: list[dict[str, Any]] = []

    for league_id, league_name in API_FOOTBALL_LEAGUES.items():
        for season in API_FOOTBALL_SEASONS:

            raw_file = get_raw_file_path(
                league_id,
                season,
            )

            if not raw_file.exists():
                raise RuntimeError(
                    "Cannot build consolidated team file because "
                    f"raw file is missing: {raw_file}"
                )

            payload = read_json_file(
                raw_file
            )

            teams = validate_teams_payload(
                payload,
                league_id=league_id,
                season=season,
            )

            for team in teams:
                all_teams.append(
                    {
                        "league_id": league_id,
                        "league_name": league_name,
                        "season": season,
                        "data": team,
                    }
                )

    if not all_teams:
        raise RuntimeError(
            "Consolidated team dataset would be empty."
        )

    write_json_atomic(
        data=all_teams,
        file_path=CONSOLIDATED_OUTPUT_FILE,
    )

    return len(
        all_teams
    )


# -----------------------------------------------------------------------------
# Load API credential
# -----------------------------------------------------------------------------
# Credentials are retrieved from Databricks Secrets and never stored directly
# in notebook source code.

api_key = dbutils.secrets.get(
    scope=API_FOOTBALL_SECRET_SCOPE,
    key=API_FOOTBALL_SECRET_KEY,
)


# -----------------------------------------------------------------------------
# Ingestion state
# -----------------------------------------------------------------------------

requests_made_this_run = 0
files_downloaded_this_run = 0
files_reused = 0

stop_for_daily_budget = False


# -----------------------------------------------------------------------------
# Fetch teams
# -----------------------------------------------------------------------------
# Processing order is deterministic:
#
#   league -> season
#
# A valid existing raw file is treated as a completed checkpoint and does not
# generate another API request.

with APIFootballClient(
    api_key=api_key,
    base_url=API_FOOTBALL_BASE_URL,
    min_request_interval_seconds=API_FOOTBALL_MIN_REQUEST_INTERVAL_SECONDS,
) as client:

    for league_id, league_name in API_FOOTBALL_LEAGUES.items():

        if stop_for_daily_budget:
            break

        for season in API_FOOTBALL_SEASONS:

            if stop_for_daily_budget:
                break

            print(
                "\n"
                "------------------------------------------------------------"
            )

            print(
                f"Processing {league_name} "
                f"(league_id={league_id}), season={season}"
            )

            raw_file = get_raw_file_path(
                league_id,
                season,
            )


            # -----------------------------------------------------------------
            # Reuse existing extraction
            # -----------------------------------------------------------------

            if league_season_is_complete(
                league_id,
                season,
            ):

                payload = read_json_file(
                    raw_file
                )

                teams = validate_teams_payload(
                    payload,
                    league_id=league_id,
                    season=season,
                )

                files_reused += 1

                print(
                    f"Already complete: {len(teams)} teams. "
                    "No API request required."
                )

                continue


            # -----------------------------------------------------------------
            # Protect configured request budget
            # -----------------------------------------------------------------

            if requests_made_this_run >= MAX_REQUESTS_PER_RUN:

                print(
                    "\nConfigured daily request budget reached."
                )

                stop_for_daily_budget = True
                break


            # -----------------------------------------------------------------
            # Protect observed API quota
            # -----------------------------------------------------------------
            # API-Football's response headers are authoritative because other
            # notebooks or manual requests may already have consumed quota.

            if (
                client.daily_remaining is not None
                and client.daily_remaining
                <= API_FOOTBALL_DAILY_REQUEST_RESERVE
            ):

                print(
                    "\nAPI-Football daily quota reached the "
                    "configured safety reserve."
                )

                stop_for_daily_budget = True
                break


            # -----------------------------------------------------------------
            # Fetch one league-season
            # -----------------------------------------------------------------

            print(
                "Fetching teams..."
            )

            payload = client.get(
                endpoint=ENDPOINT,
                params={
                    "league": league_id,
                    "season": season,
                },
            )

            requests_made_this_run += 1


            # -----------------------------------------------------------------
            # Validate before writing
            # -----------------------------------------------------------------

            teams = validate_teams_payload(
                payload,
                league_id=league_id,
                season=season,
            )


            # -----------------------------------------------------------------
            # Persist raw response atomically
            # -----------------------------------------------------------------
            # The complete API envelope is stored rather than only the team
            # records, preserving:
            #
            #   get
            #   parameters
            #   errors
            #   results
            #   paging
            #   response
            #
            # This gives the raw layer better lineage and makes debugging
            # source behavior easier.

            write_json_atomic(
                data=payload,
                file_path=str(raw_file),
            )

            files_downloaded_this_run += 1

            print(
                f"Saved {len(teams)} teams."
            )

            print(
                "API daily requests remaining: "
                f"{client.daily_remaining}"
            )


# -----------------------------------------------------------------------------
# Run summary
# -----------------------------------------------------------------------------

total_combinations = (
    len(API_FOOTBALL_LEAGUES)
    * len(API_FOOTBALL_SEASONS)
)

completed_combinations = sum(
    1
    for league_id in API_FOOTBALL_LEAGUES
    for season in API_FOOTBALL_SEASONS
    if league_season_is_complete(
        league_id,
        season,
    )
)

print(
    "\n"
    "============================================================"
)

print("Team ingestion summary")

print(
    "============================================================"
)

print(
    f"Requests made this run: {requests_made_this_run}"
)

print(
    f"New league-season files downloaded: "
    f"{files_downloaded_this_run}"
)

print(
    f"Existing files reused: {files_reused}"
)

print(
    f"Completed league-season combinations: "
    f"{completed_combinations}/{total_combinations}"
)


# -----------------------------------------------------------------------------
# Build consolidated raw dataset
# -----------------------------------------------------------------------------
# Only replace the consolidated team file when the complete configured
# extraction is available.
#
# This prevents Bronze from accidentally consuming a partially refreshed
# historical dataset.

if all_extractions_complete():

    record_count = build_consolidated_team_file()

    print(
        "\nAll configured team extractions are complete."
    )

    print(
        f"Consolidated {record_count} team-season records."
    )

    print(
        f"Consolidated raw file: {CONSOLIDATED_OUTPUT_FILE}"
    )

else:

    print(
        "\nTeam extraction is not complete yet."
    )

    print(
        "Run this notebook again after additional API quota is available."
    )

    print(
        "Completed league-season files will be reused automatically."
    )